# PropHunt + tropidist playground ([[18,4,4]] and custom CSS)

Interactive notebook to:
1. load or edit your own `Hx` / `Hz` (and optional `Lx` / `Lz`),
2. build an interleaved PropHunt CNOT schedule,
3. optionally optimize with PropHunt,
4. export timing matrices + Stim,
5. certify circuit distance with tropidist.

**Kernel:** select **PropHunt (QC tools)** (registered from `tools/PropHunt/.venv`).  
tropidist still runs via its own venv through helpers in `prophunt_tropidist.py`.

In [3]:
from pathlib import Path
import json
import pickle

import numpy as np
import matplotlib.pyplot as plt

import prophunt_tropidist as pt
from prop_hunt.prop_hunt import PropHuntCompiler

HERE = Path('.').resolve()
OUT = HERE / 'prophunt_playground_out'
OUT.mkdir(exist_ok=True)

print('HERE     :', HERE)
print('PropHunt :', pt.PROPHUNT)
print('tropidist:', pt.TROPIDIST_PY)
print('outdir   :', OUT)

ModuleNotFoundError: No module named 'parse'

## 1. Input check matrices

Use the local `hx.npy` / `hz.npy`, load your own files, or paste arrays below.
Entries should be `{0,1}` (nonzero values are thresholded to support).

In [ ]:
# Option A: load from disk (default [[18,4,4]] checks in this folder)
hx = pt.as_binary(pt.load_matrix(HERE / 'hx.npy'), 'Hx')
hz = pt.as_binary(pt.load_matrix(HERE / 'hz.npy'), 'Hz')

lx_path, lz_path = HERE / 'lx.npy', HERE / 'lz.npy'
lx = pt.as_binary(pt.load_matrix(lx_path), 'Lx') if lx_path.exists() else None
lz = pt.as_binary(pt.load_matrix(lz_path), 'Lz') if lz_path.exists() else None

# Option B: override with your own arrays (uncomment)
# hx = np.array([
#     [1,0,1,1,0,0,0,0,0,1,1,0,0,0,0,1,0,0],
#     # ...
# ], dtype=int)
# hz = hx.copy()  # example only
# lx = lz = None  # let qldpc derive logicals

print('Hx', hx.shape, 'row weights', hx.sum(1).tolist())
print('Hz', hz.shape, 'row weights', hz.sum(1).tolist())
print('Lx', None if lx is None else lx.shape, 'Lz', None if lz is None else lz.shape)
hx, hz

## 2. Build PropHunt schedule (coloration / interleaved init)

In [ ]:
graph = pt.build_graph(hx, hz, lx, lz)
nx, nz, n = hx.shape[0], hz.shape[0], hx.shape[1]

zsch, xsch = pt.graph_to_timing_matrices(graph, nx, nz, n)
depth = len(graph.to_sched())
print(f'interleaved depth = {depth}')
print('Z schedule (rows=Z-checks, cols=data):')
print(zsch)
print('X schedule:')
print(xsch)

In [ ]:
def show_schedule(zsch, xsch, title='schedule'):
    fig, axs = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    for ax, mat, name in ((axs[0], zsch, 'Z'), (axs[1], xsch, 'X')):
        im = ax.imshow(mat, aspect='auto', interpolation='nearest')
        ax.set_title(f'{name} timing')
        ax.set_xlabel('data qubit')
        ax.set_ylabel('check')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                if mat[i, j]:
                    ax.text(j, i, str(int(mat[i, j])), ha='center', va='center', fontsize=7, color='w')
    fig.suptitle(title)
    plt.show()

show_schedule(zsch, xsch, title=f'baseline (depth {depth})')

## 3. (Optional) PropHunt optimize

Set `RUN_OPTIMIZE = True` to search. Each unique snapshot is kept in `family`.

In [ ]:
RUN_OPTIMIZE = False  # flip to True when you want a search
DISTANCE = 4
MAX_ITER = 5
SAMPLES = 100
WORKERS = 4

family = [(0, graph)]  # (snapshot_index, PropagationGraph)

if RUN_OPTIMIZE:
    hist = PropHuntCompiler(graph).compile(
        max_iter=MAX_ITER,
        distance=DISTANCE,
        num_samples=SAMPLES,
        num_workers=WORKERS,
    )
    family = pt.unique_family(hist)
    print(f'history={len(hist)}, unique schedulable={len(family)}')
else:
    print('optimize skipped; family = baseline only')

for i, g in family:
    print(f'  snapshot {i}: depth={len(g.to_sched())}')

## 4. Verify circuit distance with tropidist

Certifies Z and X memory circuits for each schedule in `family`.
Use a short `TIME_LIMIT` while exploring; raise it for harder instances.

In [ ]:
ROUNDS = 4
P = 1e-3
THREADS = 4
TIME_LIMIT = 60.0
BASES = ('Z', 'X')

results = []
for snap_i, g in family:
    label = f'sched_{snap_i:03d}' + ('_baseline' if snap_i == 0 and not RUN_OPTIMIZE else '')
    print(f'\n=== {label} ===')
    rec = pt.export_and_certify(
        g, label, OUT, nx, nz, n,
        rounds=ROUNDS, p=P, threads=THREADS, time_limit=TIME_LIMIT,
        certify_bases=BASES,
    )
    results.append(rec)
    print('circuit distance interval:', rec['circuit_distance_interval'])

summary = {
    'hx_shape': list(hx.shape),
    'hz_shape': list(hz.shape),
    'family': results,
}
(OUT / 'summary.json').write_text(json.dumps(summary, indent=2))
print('\nwrote', OUT / 'summary.json')

In [ ]:
# Compact table of certified bounds
rows = []
for r in results:
    zb = r['bases'].get('Z', {})
    xb = r['bases'].get('X', {})
    clb, cub = r['circuit_distance_interval']
    rows.append({
        'label': r['label'],
        'depth': r['depth'],
        'Z': f"[{zb.get('lower_bound')}, {zb.get('upper_bound')}]",
        'X': f"[{xb.get('lower_bound')}, {xb.get('upper_bound')}]",
        'circuit': f'[{clb}, {cub}]',
    })

import pandas as pd
df = pd.DataFrame(rows)
df

## 5. Inspect one schedule in detail

Pick a label from the table above (e.g. `sched_000_baseline`).

In [ ]:
LABEL = results[0]['label']  # change me

z = np.load(OUT / f'{LABEL}_zschedule.npy')
x = np.load(OUT / f'{LABEL}_xschedule.npy')
show_schedule(z, x, title=LABEL)

tosched = pickle.load(open(OUT / f'{LABEL}_tosched.pkl', 'rb'))
print(f'{LABEL} PropHunt moments ({len(tosched)}):')
for t, (gate, pairs) in tosched.items():
    print(f'  t={t}: {gate} {pairs}')

cert_z = json.loads((OUT / f'{LABEL}_Z.json').read_text())
print('\nZ certificate:', {k: cert_z[k] for k in ('status', 'lower_bound', 'upper_bound', 'seconds', 'witness')})

## Tips

- Custom checks: edit the Option B cell, keep `Hx`/`Hz` as `(n_checks, n_data)` binary matrices with the same `n_data`.
- If you skip `Lx`/`Lz`, qldpc derives logicals from the CSS code.
- Artifacts land in `prophunt_playground_out/` (`*_zschedule.npy`, `*_xschedule.npy`, `*.stim`, tropidist `*.json`).
- For a non-interactive batch run, use `./run_prophunt_tropidist.sh ...` instead.